In [ ]:
import chess
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# 1. Neural Network: A simple CNN that "sees" the board
class ChessNet(nn.Module):
    def __init__(self):
        super(ChessNet, self).__init__()
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Sequential(nn.Linear(64 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 1))

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = x.view(-1, 64 * 8 * 8)
        return self.fc1(x)

# 2. Board Encoding: Convert chess board to a 12-layer tensor (6 pieces x 2 colors)
def encode_board(board):
    matrix = np.zeros((12, 8, 8), dtype=np.float32)
    piece_map = {'P':0, 'N':1, 'B':2, 'R':3, 'Q':4, 'K':5,
                 'p':6, 'n':7, 'b':8, 'r':9, 'q':10, 'k':11}
    for square, piece in board.piece_map().items():
        row, col = divmod(square, 8)
        matrix[piece_map[piece.symbol()], row, col] = 1
    return torch.from_numpy(matrix).unsqueeze(0)

# 3. Simple Training Loop (The 30-Minute Version)
def train():
    model = ChessNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    for episode in range(50): # 50 quick games
        board = chess.Board()
        while not board.is_game_over():
            legal_moves = list(board.legal_moves)
            move_scores = []
            
            # Evaluate all legal moves
            for move in legal_moves:
                board.push(move)
                score = model(encode_board(board))
                move_scores.append(score)
                board.pop()
            
            # Epsilon-greedy: Choose best move or random
            if random.random() > 0.1:
                move = legal_moves[torch.argmax(torch.cat(move_scores))]
            else:
                move = random.choice(legal_moves)

            # Reward logic: Simplified for speed (Capturing pieces)
            reward = 0
            if board.is_capture(move): reward = 1 
            
            board.push(move)
            
            # Update Model
            target = torch.tensor([[reward]], dtype=torch.float32)
            current_pred = model(encode_board(board))
            loss = loss_fn(current_pred, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        print(f"Episode {episode} complete.")

train()

In [ ]:
import chess

def interact_with_engine(model):
    board = chess.Board()
    print("\n--- DRL Engine Active ---")
    
    while not board.is_game_over():
        # 1. Take User Input
        user_move_str = input("\nEnter your move (e.g., e2e4): ")
        
        try:
            user_move = chess.Move.from_uci(user_move_str)
            if user_move in board.legal_moves:
                board.push(user_move)
            else:
                print("Illegal move! Try again.")
                continue
        except:
            print("Invalid format. Use UCI (e.g., g1f3).")
            continue

        # 2. Engine Logic: Predict the best response
        legal_moves = list(board.legal_moves)
        best_move = None
        max_value = -float('inf')

        for move in legal_moves:
            board.push(move)
            # This is the "Deep" part: passing the board into the trained network
            with torch.no_grad():
                board_tensor = encode_board(board) # Uses the function from training
                value = model(board_tensor).item()
            
            if value > max_value:
                max_value = value
                best_move = move
            board.pop()

        # 3. Output the result
        print(f"AI Response: {best_move}")
        board.push(best_move)
        print("\nCurrent Board State:")
        print(board)

# Call the function with your trained model
interact_with_engine(model)

In [ ]:
import chess
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# --- 1. THE BRAIN: Neural Network Architecture ---
class ChessNet(nn.Module):
    def __init__(self):
        super(ChessNet, self).__init__()
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 1) # Outputs the 'Value' of the board
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = x.view(-1, 64 * 8 * 8)
        return self.fc1(x)

# --- 2. THE TRANSLATOR: Convert Board to Tensors ---
def encode_board(board):
    matrix = np.zeros((12, 8, 8), dtype=np.float32)
    piece_map = {'P':0, 'N':1, 'B':2, 'R':3, 'Q':4, 'K':5,
                 'p':6, 'n':7, 'b':8, 'r':9, 'q':10, 'k':11}
    for square, piece in board.piece_map().items():
        row, col = divmod(square, 8)
        matrix[piece_map[piece.symbol()], row, col] = 1
    return torch.from_numpy(matrix).unsqueeze(0)

# --- 3. THE TRAINING: DRL Loop ---
def train_model():
    model = ChessNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    print("Training DRL Agent (50 Episodes)...")
    for episode in range(50):
        board = chess.Board()
        while not board.is_game_over() and board.fullmove_number < 50:
            legal_moves = list(board.legal_moves)
            move = random.choice(legal_moves)
            
            # Simple Reward: +1 for captures
            reward = 1.0 if board.is_capture(move) else 0.0
            board.push(move)
            
            # Train model to predict the reward
            target = torch.tensor([[reward]], dtype=torch.float32)
            prediction = model(encode_board(board))
            loss = loss_fn(prediction, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        if episode % 10 == 0:
            print(f"Episode {episode} processed.")
    
    print("Training Complete!\n")
    return model

# --- 4. THE INTERFACE: Supervisor Input Logic ---
def play_vs_ai(trained_model):
    board = chess.Board()
    print("AI IS READY. Input your moves in UCI format (e.g., e2e4, g1f3).")
    
    while not board.is_game_over():
        print("\nCurrent Board State:")
        print(board)
        
        # User Input
        user_input = input("\nYour Move: ")
        try:
            move = chess.Move.from_uci(user_input)
            if move in board.legal_moves:
                board.push(move)
            else:
                print("Illegal move! Try again.")
                continue
        except:
            print("Invalid format! Use e2e4 style.")
            continue

        if board.is_game_over(): break

        # Engine Logic (Inference)
        best_move = None
        max_val = -float('inf')
        
        for ai_move in board.legal_moves:
            board.push(ai_move)
            with torch.no_grad():
                val = trained_model(encode_board(board)).item()
            if val > max_val:
                max_val = val
                best_move = ai_move
            board.pop()
        
        board.push(best_move)
        print(f"---> AI played: {best_move}")

# --- EXECUTION ---
if __name__ == "__main__":
    my_trained_model = train_model()
    play_vs_ai(my_trained_model)